# Data Source

The dataset consists of two CSV files exported from Salesforce.  
They were generated as a result of an email campaign with A/B testing (sent vs. not sent).  
Each file contains information about customers and their subscription behavior.

## Imports
This section imports the necessary Python libraries for data processing and analysis.

In [58]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

## Data Loading
In this section, two CSV files from the A/B test campaign are loaded and combined into a single dataset.

### Data note

Due to Salesforce limitations, campaign data does not include the actual cancellation request date.

An additional report was used to extract **"Kündigung erfasst am"**, which reflects the true timing of churn.

Data was merged via `customer_id`.

`cancel_date` is used for analysis instead of `end_date` to avoid time lag bias (~1 month delay between request and contract end).

In [70]:
base_path = Path("../Data/raw")

read_kwargs = {
    "sep": ";",
    "dtype": {"SAP GP Nummer": "string"}
}

# load campaign datasets (treatment / control)
df1 = pd.read_csv(base_path / "Kampagnenmitglieder_nein_raw.csv", **read_kwargs)
df2 = pd.read_csv(base_path / "Kampagnenmitglieder_ja_raw.csv", **read_kwargs)

# load cancellation request data ("Kündigung erfasst am")
df_kuend = pd.read_csv(base_path / "Kuendigung erfasst am-2026-04-22-15-57-42.csv", **read_kwargs)

# keep relevant columns from campaign data
relevant_cols = [
    "SAP GP Nummer",
    "letzter Abo - Auftrag: Auftrag von",
    "letzter Abo - Auftrag: Auftrag bis",
    "letzter Abo - Auftrag: Kündigungsgrund Code"
]

df1_clean = df1[relevant_cols].copy()
df2_clean = df2[relevant_cols].copy()

# add treatment indicator
df1_clean["treatment"] = 0
df2_clean["treatment"] = 1

# combine groups
df = pd.concat([df1_clean, df2_clean], ignore_index=True)

# rename columns
df = df.rename(columns={
    "SAP GP Nummer": "customer_id",
    "letzter Abo - Auftrag: Auftrag von": "start_date",
    "letzter Abo - Auftrag: Auftrag bis": "end_date",
    "letzter Abo - Auftrag: Kündigungsgrund Code": "cancel_reason"
})

# keep relevant columns from cancellation dataset
df_kuend_clean = df_kuend[[
    "SAP GP Nummer",
    "letzter Abo - Auftrag: Kündigung erfasst am"
]].copy()

df_kuend_clean = df_kuend_clean.rename(columns={
    "SAP GP Nummer": "customer_id",
    "letzter Abo - Auftrag: Kündigung erfasst am": "cancel_date"
})

for col in df.columns:
    print(col)

for col in df_kuend_clean.columns:
    print(col)

    # unify key type
df["customer_id"] = df["customer_id"].astype(str).str.strip()
df_kuend_clean["customer_id"] = df_kuend_clean["customer_id"].astype(str).str.strip()

# convert dates
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce", dayfirst=True)
df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce", dayfirst=True)
df_kuend_clean["cancel_date"] = pd.to_datetime(
    df_kuend_clean["cancel_date"], errors="coerce", dayfirst=True
)

# if duplicates → keep earliest cancel per customer
df_kuend_clean = (
    df_kuend_clean
    .sort_values("cancel_date")
    .drop_duplicates("customer_id", keep="first")
)

# merge
df = df.merge(df_kuend_clean, on="customer_id", how="left")

df.head()
print(df["customer_id"].dtype)
print(df_kuend_clean["customer_id"].dtype)

df["customer_id"].head(10).tolist()

customer_id
start_date
end_date
cancel_reason
treatment
customer_id
cancel_date
str
str


['0018640485',
 '0018703242',
 '0019173245',
 '0019173659',
 '0019173423',
 '0019173333',
 '0019173437',
 '0019173673',
 '0017296969',
 '0018114828']

# Data Cleaning

In [71]:
print(df["cancel_date"].notna().sum())

412


In [72]:
# define observation window for post-campaign churn
start = pd.Timestamp("2026-03-19")
end = pd.Timestamp("2026-04-19")

# create churn variable: 1 if subscription ended within the observation window
df["churn"] = (
    df["cancel_date"].notna() &
    (df["cancel_date"] >= start) &
    (df["cancel_date"] <= end)
).astype(int)

# overall churn rate (%)
overall_churn = df["churn"].mean() * 100

# overall churn by treatment group (%)
churn_by_group = df.groupby("treatment")["churn"].mean() * 100
churn_by_group.index = ["Not sent", "Sent"]

# KA churn based on the same base: all customers in each group
ka_churn_by_group = df.groupby("treatment").apply(
    lambda x: ((x["churn"] == 1) & (x["cancel_reason"] == "KA")).mean() * 100
)
ka_churn_by_group.index = ["Not sent", "Sent"]

# share of KA among all churned customers (%)
ka_share_among_churned = (
    (df.loc[df["churn"] == 1, "cancel_reason"] == "KA").mean() * 100
)

# print results
print(f"Overall churn: {overall_churn:.4f}%\n")

print("Overall churn by treatment group (%):")
print(churn_by_group.round(4))

print("\nKA churn by treatment group (% of all customers):")
print(ka_churn_by_group.round(4))

print(f"\nShare of KA among all churned customers: {ka_share_among_churned:.4f}%")

Overall churn: 1.3303%

Overall churn by treatment group (%):
Not sent    1.4845
Sent        1.2960
Name: churn, dtype: float64

KA churn by treatment group (% of all customers):
Not sent    0.3663
Sent        0.3294
dtype: float64

Share of KA among all churned customers: 25.2632%


In [78]:
!pip install statsmodels
from statsmodels.stats.proportion import proportions_ztest

def test_segment(df, segment):
    sub = df[df["tenure_group"] == segment]

    n_control = (sub["treatment"] == 0).sum()
    n_treat = (sub["treatment"] == 1).sum()

    churn_control = sub.loc[sub["treatment"] == 0, "churn"].sum()
    churn_treat = sub.loc[sub["treatment"] == 1, "churn"].sum()

    stat, pval = proportions_ztest(
        [churn_treat, churn_control],
        [n_treat, n_control]
    )

    uplift = (churn_treat / n_treat) - (churn_control / n_control)

    return uplift, pval

for seg in df["tenure_group"].dropna().unique():
    uplift, pval = test_segment(df, seg)
    print(seg, f"uplift={uplift:.4%}, p={pval:.4f}")

  Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl (9.5 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [st


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'statsmodels'

In [73]:
df["tenure_days"] = (pd.Timestamp("2026-03-19") - df["start_date"]).dt.days

df["tenure_group"] = pd.cut(
    df["tenure_days"],
    bins=[0, 90, 365, 2000],
    labels=["new", "mid", "old"]
)

df.groupby(["tenure_group", "treatment"])["churn"].mean().unstack()

treatment,0,1
tenure_group,,
new,0.026930,0.020011
mid,0.010811,0.008535
old,0.011834,0.011806


In [74]:
# filter only customers who churned in the campaign window
df_window = df[
    (df["churn"] == 1)
].copy()

# count per cancel_reason (only in this month)
counts = df_window["cancel_reason"].value_counts()

# one example customer per reason
examples = (
    df_window[["cancel_reason", "customer_id"]]
    .dropna(subset=["cancel_reason"])
    .drop_duplicates(subset=["cancel_reason"])
    .set_index("cancel_reason")
)

# combine
df_reason_map = (
    counts.to_frame("count")
    .join(examples)
    .reset_index()
    .rename(columns={"index": "cancel_reason"})
)
df_window.groupby(["treatment", "cancel_reason"]).size().reset_index(name="count")

df_reason_map


,cancel_reason,count,customer_id
0,PRE,106,0016113692
1,KA,96,0018684395
2,TOD,22,0019291565
3,KÜR,22,0018067004
4,KIN,21,0019300717
5,KZA,17,0019493127
6,AME,13,0019504428
7,KZE,13,0019239475
8,INH,11,0019335515
9,AKP,10,0016102967


In [75]:
# define observation window
start = pd.Timestamp("2026-03-19")
end = pd.Timestamp("2026-04-18")

# filter only churned customers in this window
df_window = df[
    (df["cancel_date"].notna()) &
    (df["cancel_date"] >= start) &
    (df["cancel_date"] <= end)
].copy()

# group by treatment and cancel_reason (counts)
result = (
    df_window
    .groupby(["treatment", "cancel_reason"])
    .size()
    .reset_index(name="count")
)

pivot_pct = (
    df_window
    .groupby(["treatment", "cancel_reason"])
    .size()
    .groupby(level=0)
    .apply(lambda x: x / x.sum() * 100)
    .unstack(fill_value=0)
    .T
)

pivot_pct.rename(columns={0: "Not sent", 1: "Sent"}).round(2)

treatment,Not sent,Sent
treatment,Not sent,Sent
cancel_reason,,
AKP,5.33,1.68
AME,0.00,4.04
AZE,1.33,0.00
BEF,4.00,1.01
BON,1.33,0.34
DPZ,0.00,0.34
FIN,2.67,1.68
INH,4.00,2.69


In [76]:
churn_end = (
    df["end_date"].notna() &
    (df["end_date"] >= start) &
    (df["end_date"] <= end)
).astype(int)

churn_cancel = (
    df["cancel_date"].notna() &
    (df["cancel_date"] >= start) &
    (df["cancel_date"] <= end)
).astype(int)

print((churn_end != churn_cancel).sum())
df["lag_days"] = (df["end_date"] - df["cancel_date"]).dt.days
df["lag_days"].describe()

521


count    411.000000
mean      87.518248
std      130.139031
min       -1.000000
25%       23.000000
50%       29.000000
75%       85.000000
max      680.000000
Name: lag_days, dtype: float64